In [0]:
#create a variable for the dataset
landing_file='/Volumes/nhsae/raw/landing/ae_2026_03.csv'

In [0]:
#read the dataset 
df=spark.read.format('csv').option('header',True).load(landing_file)

In [0]:
#display dataset for initial exploration
display(df)

In [0]:
#check the number of rows in dataset
df.count()

In [0]:
#print the schema
df.printSchema()

In [0]:
# Schema contract: exact NHS column name to a  clean snake_case name
COLUMN_MAP = {
    "Period": "period",
    "Org Code": "org_code",
    "Parent Org": "parent_org",
    "Org name": "org_name",
    "A&E attendances Type 1": "att_t1_unbooked",
    "A&E attendances Type 2": "att_t2_unbooked",
    "A&E attendances Other A&E Department": "att_oth_unbooked",
    "A&E attendances Booked Appointments Type 1": "att_t1_booked",
    "A&E attendances Booked Appointments Type 2": "att_t2_booked",
    "A&E attendances Booked Appointments Other Department": "att_oth_booked",
    "Attendances over 4hrs Type 1": "o4_t1_unbooked",
    "Attendances over 4hrs Type 2": "o4_t2_unbooked",
    "Attendances over 4hrs Other Department": "o4_oth_unbooked",
    "Attendances over 4hrs Booked Appointments Type 1": "o4_t1_booked",
    "Attendances over 4hrs Booked Appointments Type 2": "o4_t2_booked",
    "Attendances over 4hrs Booked Appointments Other Department": "o4_oth_booked",
    "Patients who have waited 4-12 hs from DTA to admission": "dta_4_12hr",
    "Patients who have waited 12+ hrs from DTA to admission": "dta_12hr_plus",
    "Emergency admissions via A&E - Type 1": "adm_ae_t1",
    "Emergency admissions via A&E - Type 2": "adm_ae_t2",
    "Emergency admissions via A&E - Other A&E department": "adm_ae_oth",
    "Other emergency admissions": "adm_other",
}
EXPECTED_COLUMNS = list(COLUMN_MAP.keys())
print(len(EXPECTED_COLUMNS))  # should be 22


In [0]:
#File level integrity check
missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
unexpected = [c for c in df.columns if c not in EXPECTED_COLUMNS]
print("Missing:", missing)
print("Unexpected:", unexpected)
#both lists should return empty.In the pipeline, a non empty list means the 
#file goes into quarantine

In [0]:
#Split off the TOTAL row
from pyspark.sql import functions as F

total_df = df.filter(F.col("Org Code") == "TOTAL")
org_df = df.filter((F.col("Org Code") != "TOTAL") | F.col("Org Code").isNull())
print("Total rows:", total_df.count(), "| Org rows:", org_df.count())  # expect 1 | 199

In [0]:
#Rename,cast types and parse date
#try_cast,try_date allows spark to try working on the item,
#but if it fails, returns null instead of crashing. it's a safety net'''''

TEXT_COLS = ["period", "org_code", "parent_org", "org_name"]
NUMERIC_COLS = [v for v in COLUMN_MAP.values() if v not in TEXT_COLS]

renamed = org_df.select([F.col(src).alias(dst) for src, dst in COLUMN_MAP.items()])

typed = renamed.select(
    *[F.trim(F.col(c)).alias(c) for c in TEXT_COLS],
    *[F.expr(f"try_cast({c} AS INT)").alias(c) for c in NUMERIC_COLS],
    *[F.col(c).alias(f"{c}_raw") for c in NUMERIC_COLS],
).withColumn(
    "period_date",
        F.expr("try_to_date(initcap(regexp_extract(period, '([A-Za-z]+-[0-9]{4})$', 1)), 'MMMM-yyyy')")
)

display(typed.select("period", "period_date").distinct())

In [0]:
print(renamed.columns)

In [0]:
# Did any conversion fail? Failure means there was a value, but it wasnt a number
failed = {
    c: typed.filter(F.col(f"{c}_raw").isNotNull() & F.col(c).isNull()).count()
    for c in NUMERIC_COLS
}
print({c: n for c, n in failed.items() if n > 0} or "All numeric values converted")

In [0]:
# Data Quality Checks
from functools import reduce
from pyspark.sql import Window

# (attendances, over_4hr) column pairs
PAIRS = [(f"att_{t}_{r}", f"o4_{t}_{r}") for t in ["t1", "t2", "oth"] for r in ["unbooked", "booked"]]

w = Window.partitionBy("org_code", "period_date")
checked = typed.withColumn("dup_count", F.count("*").over(w))

RULES = {
    "missing_org_code": F.col("org_code").isNull() | (F.col("org_code") == ""),
    "failed_type_conversion": reduce(lambda a, b: a | b,
        [F.col(f"{c}_raw").isNotNull() & F.col(c).isNull() for c in NUMERIC_COLS]),
    "negative_value": F.least(*[F.col(c) for c in NUMERIC_COLS]) < 0,
    "over_4hr_exceeds_attendances": reduce(lambda a, b: a | b,
        [F.col(o) > F.col(a) for a, o in PAIRS]),
    "duplicate_org_period": F.col("dup_count") > 1,
}

checked = checked.withColumn(
    "dq_reason",
    F.concat_ws("; ", *[F.when(cond, F.lit(name)) for name, cond in RULES.items()])
)

valid = checked.filter(F.col("dq_reason") == "")
invalid = checked.filter(F.col("dq_reason") != "")

n_total, n_bad = checked.count(), invalid.count()
print(f"Valid: {n_total - n_bad} | Invalid: {n_bad} | {n_bad / n_total:.1%} invalid")
display(invalid.select("org_code", "org_name", "dq_reason"))

In [0]:
#silver ae by type
#STACK(6,..) unpivots. it turns each wide row into 6 long rows, taking 4 values
#at a time:type,route,attendances, and over_4hr, in a wide-->long design
by_type = valid.select(
    "period_date", "org_code",
    F.expr("""stack(6,
        'Type 1', 'unbooked', att_t1_unbooked, o4_t1_unbooked,
        'Type 2', 'unbooked', att_t2_unbooked, o4_t2_unbooked,
        'Other',  'unbooked', att_oth_unbooked, o4_oth_unbooked,
        'Type 1', 'booked',   att_t1_booked,   o4_t1_booked,
        'Type 2', 'booked',   att_t2_booked,   o4_t2_booked,
        'Other',  'booked',   att_oth_booked,  o4_oth_booked
    ) AS (dept_type, route, attendances, over_4hr)""")
)
print(by_type.count())  # valid orgs x 6
display(by_type.orderBy("org_code", "route", "dept_type"))

In [0]:
#silver ae_org monthly
''''In spark, number+ null=null,so if one of the three admission column is ever empty,
the total becomes null rather than ignoring the gap. A data qua;ity check'''''

org_monthly = valid.select(
    "period_date", "org_code", "org_name", "parent_org",
    "dta_4_12hr", "dta_12hr_plus",
    "adm_ae_t1", "adm_ae_t2", "adm_ae_oth", "adm_other",
).withColumn(
    "emergency_admissions_ae",
    F.col("adm_ae_t1") + F.col("adm_ae_t2") + F.col("adm_ae_oth")
)
print(org_monthly.count())
display(org_monthly)

In [0]:
#Reconciliation against TOTAL row
total_typed = total_df.select([
    F.expr(f"try_cast(`{src}` AS INT)").alias(dst)
    for src, dst in COLUMN_MAP.items() if dst in NUMERIC_COLS
])

org_sums = typed.agg(*[F.sum(c).alias(c) for c in NUMERIC_COLS]).collect()[0].asDict()
total_vals = total_typed.collect()[0].asDict()

mismatches = {c: (org_sums[c], total_vals[c]) for c in NUMERIC_COLS if org_sums[c] != total_vals[c]}
print(mismatches or "Reconciled: organisation rows sum exactly to the TOTAL row")

In [0]:
#Building the first silver tables...org by departments, and org by months
spark.sql("""
CREATE TABLE IF NOT EXISTS nhsae.silver.ae_by_type (
  period_date DATE, org_code STRING, dept_type STRING, route STRING,
  attendances INT, over_4hr INT,
  source_file STRING, ingested_at TIMESTAMP
) COMMENT 'A&E attendances and 4-hour breaches. Grain: org x month x department type x route'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS nhsae.silver.ae_org_monthly (
  period_date DATE, org_code STRING, org_name STRING, parent_org STRING,
  dta_4_12hr INT, dta_12hr_plus INT,
  adm_ae_t1 INT, adm_ae_t2 INT, adm_ae_oth INT, adm_other INT,
  emergency_admissions_ae INT,
  source_file STRING, ingested_at TIMESTAMP
) COMMENT 'Org-level waits from decision to admit and emergency admissions. Grain: org x month'
""")

In [0]:
#writing down the month (idempotent)
import os

source_file = os.path.basename(landing_file)       # 'ae_2026_03.csv'
period = valid.select("period_date").first()[0]    # 2026-03-01

def write_month(df, table):
    (df.withColumn("source_file", F.lit(source_file))
       .withColumn("ingested_at", F.current_timestamp())
       .write.format("delta")
       .mode("overwrite")
       .option("replaceWhere", f"period_date = '{period}'")
       .saveAsTable(table))

write_month(by_type, "nhsae.silver.ae_by_type")
write_month(org_monthly, "nhsae.silver.ae_org_monthly")
print(f"Written period {period} from {source_file}")

In [0]:
#verify and prove idempotency
display(spark.sql("""
  SELECT 'ae_by_type' AS tbl, period_date, COUNT(*) AS n FROM nhsae.silver.ae_by_type GROUP BY period_date
  UNION ALL
  SELECT 'ae_org_monthly', period_date, COUNT(*) FROM nhsae.silver.ae_org_monthly GROUP BY period_date
"""))

display(spark.sql("DESCRIBE DETAIL nhsae.silver.ae_by_type").select("location", "numFiles"))

In [0]:
#History
display(spark.sql("DESCRIBE HISTORY nhsae.silver.ae_by_type")
        .select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

In [0]:
#The gold national performance table
spark.sql("""
CREATE OR REPLACE TABLE nhsae.gold.national_by_type
COMMENT 'National A&E 4-hour performance. Grain: month x department type x route'
AS
SELECT
  period_date, dept_type, route,
  SUM(attendances) AS attendances,
  SUM(over_4hr)    AS over_4hr,
  ROUND(100 * (1 - SUM(over_4hr) / NULLIF(SUM(attendances), 0)), 2) AS pct_within_4hr
FROM nhsae.silver.ae_by_type
GROUP BY period_date, dept_type, route
""")
display(spark.table("nhsae.gold.national_by_type").orderBy("period_date", "route", "dept_type"))

In [0]:
#Gold table; performance across trust
spark.sql("""
CREATE OR REPLACE TABLE nhsae.gold.trust_monthly
COMMENT 'Provider-level 4-hour and 12-hour DTA performance. Grain: org x month'
AS
WITH att AS (
  SELECT period_date, org_code,
         SUM(attendances) AS attendances,
         SUM(over_4hr)    AS over_4hr
  FROM nhsae.silver.ae_by_type
  GROUP BY period_date, org_code
)
SELECT
  o.period_date, o.org_code, o.org_name, o.parent_org,
  CASE WHEN o.org_code RLIKE '^R[A-Z0-9]{2}$' THEN 'NHS trust' ELSE 'Other provider' END AS org_type,
  a.attendances, a.over_4hr,
  ROUND(100 * (1 - a.over_4hr / NULLIF(a.attendances, 0)), 2) AS pct_within_4hr,
  o.dta_12hr_plus, o.emergency_admissions_ae,
  ROUND(1000 * o.dta_12hr_plus / NULLIF(o.emergency_admissions_ae, 0), 1) AS dta_12hr_per_1000_adm
FROM nhsae.silver.ae_org_monthly o
JOIN att a USING (period_date, org_code)
""")
display(spark.table("nhsae.gold.trust_monthly").orderBy(F.desc("attendances")))

In [0]:
#check against published figures
display(spark.sql("""
SELECT period_date,
       SUM(attendances) AS attendances,
       ROUND(100 * (1 - SUM(over_4hr) / SUM(attendances)), 1) AS pct_within_4hr_all
FROM nhsae.gold.national_by_type
GROUP BY period_date
"""))

In [0]:
#further tests
display(spark.sql("""
SELECT 'Within 4 hrs (count)' AS metric,
       CAST(SUM(attendances) - SUM(over_4hr) AS STRING) AS my_work, '1,875,443' AS published
FROM nhsae.gold.national_by_type
UNION ALL
SELECT 'Type 1 % within 4 hrs',
       CAST(ROUND(100 * (1 - SUM(over_4hr) / SUM(attendances)), 1) AS STRING), '64.1'
FROM nhsae.gold.national_by_type WHERE dept_type = 'Type 1'
UNION ALL
SELECT 'Other (Type 3) % within 4 hrs',
       CAST(ROUND(100 * (1 - SUM(over_4hr) / SUM(attendances)), 1) AS STRING), '97.3'
FROM nhsae.gold.national_by_type WHERE dept_type = 'Other'
UNION ALL
SELECT 'DTA waits over 4 hrs',
       CAST(SUM(dta_4_12hr) + SUM(dta_12hr_plus) AS STRING), '123,870'
FROM nhsae.silver.ae_org_monthly
UNION ALL
SELECT 'DTA waits over 12 hrs',
       CAST(SUM(dta_12hr_plus) AS STRING), '46,665'
FROM nhsae.silver.ae_org_monthly
"""))

In [0]:
display(spark.sql("""
SELECT dept_type, route,
       SUM(attendances) - SUM(over_4hr) AS within_4hr
FROM nhsae.gold.national_by_type
GROUP BY ROLLUP(dept_type, route)
ORDER BY dept_type, route
"""))